In [1]:
import os
import shutil
import json
import torch
import time
import nltk
import numpy as np
from google.colab import drive
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import Dataset

# 1. CẤP QUYỀN CHO NUMPY (FIX LỖI UNPICKLING)
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray._reconstruct,
        np.ndarray,
        np.dtype,
    ])
    print("✅ Đã whitelist Numpy cho PyTorch.")
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override
print(f"✅ Đã ép buộc torch.load(weights_only=False) thành công.")

# 2. KẾT NỐI DRIVE & TẢI DỮ LIỆU
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')
FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_CRP_FFT"

print(">>> [1/4] Kiểm tra và tải dữ liệu...")
if not os.path.exists('spider_data'):
    !pip install -q kaggle
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)

# Tải bổ sung Spider-Realistic
if not os.path.exists('spider_data/spider-realistic.json'):
    !wget -q -O spider_data/spider-realistic.json "https://zenodo.org/records/5205322/files/spider-realistic.json?download=1"

# 3. TIỀN XỬ LÝ SCHEMA VÀ LOAD DATASET
with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[1]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

print(">>> [2/4] Đang load tập Spider-Realistic...")
val_ds = load_spider_unified("spider_data/spider-realistic.json")

# 4. LOAD MÔ HÌNH FFT TỪ GOOGLE DRIVE
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n>>> [3/4] Đang tải mô hình từ {FINAL_SAVE_PATH} lên {device}...")

tokenizer = T5Tokenizer.from_pretrained(FINAL_SAVE_PATH)
model = T5ForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH).to(device)
model.eval()

# 5. CHẠY INFERENCE VÀ ĐÁNH GIÁ
print("\n>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic...")
predictions, gold_lines = [], []

start_time = time.time()
for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

print(f"✅ Hoàn tất dự đoán trong {time.time() - start_time:.2f} giây.")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

print("\n>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

/tmp/ipykernel_9123/3861715649.py:15: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,


✅ Đã whitelist Numpy cho PyTorch.
✅ Đã ép buộc torch.load(weights_only=False) thành công.
Mounted at /content/drive
>>> [1/4] Kiểm tra và tải dữ liệu...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
100% 96.0M/96.0M [00:00<00:00, 101MB/s]

>>> [2/4] Đang load tập Spider-Realistic...

>>> [3/4] Đang tải mô hình từ /content/drive/My Drive/T5_Small_Spider_CRP_FFT lên cuda...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic...


Token indices sequence length is longer than the specified maximum sequence length for this model (868 > 512). Running this sequence through the model will result in indexing errors


✅ Hoàn tất dự đoán trong 242.79 giây.

>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:
eval_err_num:1
medium pred: SELECT name, country, age FROM singer WHERE age  (SELECT max(age) FROM singer WHERE age = 'old' INTERSECT SELECT name FROM singer WHERE age = 'old' AND age = 'old')
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:2
medium pred: SELECT T1.name, T1.year FROM singer AS T1 JOIN song_in_concert AS T2 ON T1.singer_id = T2.singer_id JOIN concert AS T3 ON T2.concert_id = T3.concert_id ORDER BY T3.age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:3
medium pred: SELECT T1.name, T1.year FROM singer AS T1 JOIN song_release_year AS T2 ON T1.singer_id = T2.singer_id JOIN concert AS T3 ON T2.concert_id = T3.concert_id WHERE T3.age = (SELECT max(age) FROM singer AS T4 JOIN singer_in_concert AS T4 ON T3.concert_id = T4.conce
medium gold: SELECT song_name ,  song_release_year FROM singer ORDE